# Agent offline evaluation: setup and test data

Resolves the canonical `aria-rm-briefing-agent` on the **admin project** (`project-admin-{suffix}`) - created in [`08-05b-01-private-banking-agent-setup.ipynb`](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb) - runs each sample query through it, captures responses with `thread_id` and `run_id`, and writes `test_data.jsonl` for the remaining evaluation notebooks.

The agent uses **6 intent-level MCP tools** (`cpb_prepare_client_briefing`, `cpb_analyze_portfolio_drift`, `cpb_find_relevant_research`, `cpb_summarize_recent_activity`, `cpb_get_client_context`, `cpb_run_query`) backed by a synthetic Contoso Private Investments knowledge base - see [08-05b](../08-05b-contoso-private-banking-mcp/) for the lab.

The lab targets the admin project (and not a spoke) so model graders in later notebooks can use the LLM deployment that lives natively on `aif-core-{suffix}` - spoke accounts have zero deployments by design.

The agent is invoked via the **Classic threads/runs path** because Foundry's back-compat `asst_*` shim makes the new-API versioned agent discoverable via `AgentsClient.list_agents()`, and Classic `runs.create_and_process` still produces the `thread_id` / `run_id` that the rest of the evaluation chain consumes.

## Dependencies

Managed via `pyproject.toml`. Run `uv sync` before opening.

In [1]:
import hashlib
import json
import os
import subprocess
import time
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import MessageRole
from dotenv import load_dotenv

## Environment

In [2]:
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ['CHAT_MODEL']

# Admin project endpoint - same derivation as 08-05-02 and 08-07-06.
SUB_ID  = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX  = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

AGENT_NAME = 'aria-rm-briefing-agent'

print(f'Project endpoint : {PROJECT_ENDPOINT}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'Agent name       : {AGENT_NAME}')

Project endpoint : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Chat model       : gpt-4.1-mini
Agent name       : aria-rm-briefing-agent


## AgentsClient

In [3]:
credential    = DefaultAzureCredential()
agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

## Load sample queries

In [4]:
lab_dir     = repo_root / '08-agents' / '08-06-agent-offline-evaluation'
sample_path = lab_dir / 'sample_test_data.jsonl'

sample_records = []
with open(sample_path) as f:
    for line in f:
        line = line.strip()
        if line:
            sample_records.append(json.loads(line))

print(f'Loaded {len(sample_records)} sample queries')
for r in sample_records:
    print(f'  * {r["query"][:80]}')

Loaded 5 sample queries
  * I have a 9am with the Berger family for their quarterly review. Brief me on thei
  * Show me the Lindemann family office portfolio drift over 5 percentage points so 
  * Anything been written recently about AI infrastructure capex that would be relev
  * Summarise what is been happening on the Riedi pension over the last 90 days. Gro
  * Get me the FINMA sustainability disclosure status for the Eichmann Foundation an


## Resolve the canonical aria-rm-briefing-agent

The agent is created (once) by [`08-05b-01-private-banking-agent-setup.ipynb`](../08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb) on the admin project, with the MCP tool wiring (`require_approval='never'`) configured at creation time. This notebook only looks it up by name.

If the agent is missing, run `08-05b-01` first.

In [ ]:
agent = next(
    (a for a in agents_client.list_agents() if a.name == AGENT_NAME),
    None,
)
if agent is None:
    raise RuntimeError(
        f"Agent '{AGENT_NAME}' not found on {PROJECT_ENDPOINT}. "
        f"Run 08-agents/08-05b-contoso-private-banking-mcp/08-05b-01-private-banking-agent-setup.ipynb first."
    )
print(f"Using agent: {agent.name}  (id={agent.id})")

# Build MCP toolset for run-time approval mode. The agent's tool definition already
# carries `require_approval='never'`, but supplying it again via ToolSet is harmless
# and explicit.
from azure.ai.agents.models import McpTool, ToolSet
_mcp_def = next(t for t in agent.tools if t['type'] == 'mcp')
_mcp_tool = McpTool(server_label=_mcp_def['server_label'], server_url=_mcp_def['server_url'])
_mcp_tool.set_approval_mode('never')
mcp_toolset = ToolSet()
mcp_toolset.add(_mcp_tool)

## Run agent against each query

In [6]:
def run_agent_query(client, agent_id, query, toolset=None):
    thread = client.threads.create()
    client.messages.create(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=query,
    )
    run = client.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent_id,
        toolset=toolset,
    )

    status_str = str(run.status).split('.')[-1].upper()
    if status_str != 'COMPLETED' or run.last_error:
        raise RuntimeError(
            f"Agent run did not complete cleanly.\n"
            f"  status     : {run.status}\n"
            f"  last_error : {run.last_error}\n"
            f"  thread_id  : {thread.id}\n"
            f"  run_id     : {run.id}"
        )

    response_text = ''
    for msg in client.messages.list(thread_id=thread.id):
        if msg.role == MessageRole.AGENT:
            for content in msg.content:
                if hasattr(content, 'text'):
                    response_text = content.text.value
                    break
            break
    return {'thread_id': thread.id, 'run_id': run.id, 'response': response_text}

In [ ]:
test_records = []

for i, record in enumerate(sample_records):
    print(f'[{i+1}/{len(sample_records)}] {record["query"][:60]}...')
    result = run_agent_query(agents_client, agent.id, record['query'], toolset=mcp_toolset)
    test_records.append({
        'query':        record['query'],
        'ground_truth': record['ground_truth'],
        'context':      record['context'],
        'response':     result['response'],
        'thread_id':    result['thread_id'],
        'run_id':       result['run_id'],
    })
    print(f'   response length: {len(result["response"])} chars')
    time.sleep(0.5)

## Save to test_data.jsonl

In [8]:
test_data_path = lab_dir / 'test_data.jsonl'

with open(test_data_path, 'w') as f:
    for rec in test_records:
        f.write(json.dumps(rec) + '\n')

print(f'Saved {len(test_records)} records to {test_data_path}')

Saved 5 records to <repo-root>/08-agents/08-06-agent-offline-evaluation/test_data.jsonl


## Preview first record

In [9]:
print(json.dumps(test_records[0], indent=2))

{
  "query": "I have a 9am with the Berger family for their quarterly review. Brief me on their portfolio, any drift, recent activity, anything CRM has flagged, and the 2-3 things I should be ready to talk about.",
  "ground_truth": "Briefing for Berger Family Trust (UHNW Multi-Generation, RM Anna M\u00fcller): AUM CHF 89,533,698; 1 asset-class drift flag(s); 0 concentration flag(s); 2 CRM event(s) in last 90d; 2 next-best-action(s).",
  "context": "Briefing for Berger Family Trust (UHNW Multi-Generation, RM Anna M\u00fcller): AUM CHF 89,533,698; 1 asset-class drift flag(s); 0 concentration flag(s); 2 CRM event(s) in last 90d; 2 next-best-action(s).\n\n{\"client\": {\"name\": \"Berger Family Trust\", \"segment\": \"UHNW Multi-Generation\", \"rm\": \"Anna M\u00fcller\", \"base_currency\": \"CHF\", \"aum_chf_book\": 87500000, \"aum_chf_calc\": 89533698.0, \"next_review_date\": \"2026-05-15\", \"languages\": [\"DE\", \"EN\"]}, \"meeting_purpose\": \"quarterly_review\", \"as_of\": \"2026-0